# 01 · Análisis Exploratorio de Datos (EDA)
**Fase:** Exploración general — todas las categorías CLASFINAL

Objetivos:
1. Auditar calidad del dato SIVIGILA
2. Corregir inconsistencias de forma documentada
3. Caracterizar la variable objetivo y su distribución temporal
4. Identificar sesgos que afecten la validez de los modelos

> **Contexto:** El dengue en Colombia presenta ciclos epidémicos cada 2-3 años.
> La heterogeneidad temporal en SIVIGILA puede reflejar variaciones reales
> o cambios en los criterios de clasificacion del INS.

In [ ]:
import sys
sys.path.insert(0, '../../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import joblib
from pathlib import Path
from datetime import datetime
import config
from config import (DATA_RAW, DATA_INTERIM, OUT_FIGURES, OUT_TABLES,
    CHECKPOINTS, COLORS, RANDOM_STATE, TARGET_COL,
    SIGNOS_ALARMA, SINTOMAS_GENERALES)
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi':150,'savefig.dpi':300,
    'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.titlesize':14,'axes.titleweight':'bold',
    'axes.labelsize':12,'legend.frameon':False})
print('Entorno listo')

In [ ]:
# Carga de datos
# utf-8-sig maneja el BOM de Excel, frecuente en exportaciones SIVIGILA
DATA_FILE = DATA_RAW / 'datos_dengue_bucaramanga_depurado.csv'
df = pd.read_csv(DATA_FILE, encoding='utf-8-sig', low_memory=False)
print(f'Cargado: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'  Filas   : {df.shape[0]:,}')
print(f'  Columnas: {df.shape[1]}')
print(f'  Memoria : {df.memory_usage(deep=True).sum()/1e6:.1f} MB')

## 1. Procesamiento de fechas

Las fechas SIVIGILA se almacenan como strings.
La fecha `1900-01-01` es un centinela del sistema que indica ausencia de defuncion.

In [ ]:
# Conversion de fechas
# errors='coerce' convierte fechas invalidas a NaT sin interrumpir la carga
columnas_fecha = ['FEC_NOT','FEC_CON_','INI_SIN_','FEC_HOS_','FEC_DEF_']
for col in columnas_fecha:
    df[f'{col}_DT'] = pd.to_datetime(df[col], errors='coerce')

# Reemplazar centinela 1900-01-01 por NaT
df['FEC_DEF__DT'] = df['FEC_DEF__DT'].mask(
    df['FEC_DEF__DT'].eq(pd.Timestamp('1900-01-01'))
)

# Demora en notificacion: indicador de oportunidad del sistema de vigilancia
df['DEMORA_NOTIF'] = (df['FEC_NOT_DT'] - df['INI_SIN__DT']).dt.days
validas = df['DEMORA_NOTIF'].between(0, 60)
print(f'Demora mediana: {df.loc[validas,"DEMORA_NOTIF"].median():.0f} dias')
print(f'Registros con demora valida: {validas.sum():,} ({validas.mean():.1%})')

## 2. Auditoria y corrección de la variable objetivo

Se evaluan cuatro reglas de consistencia en `PAC_HOS_`:
- **R1:** No hospitalizado pero con fecha de hospitalizacion registrada
- **R2:** Hospitalizado pero sin fecha de hospitalizacion
- **R3:** No hospitalizado pero con conducta intrahospitalaria
- **R4:** Inconsistencia estricta (R1 + R3 simultaneos)

In [ ]:
# Auditoria de consistencia
pac_hosp = pd.to_numeric(df['PAC_HOS_'], errors='coerce')
fecha_hos = pd.to_datetime(df['FEC_HOS_'], errors='coerce')
conducta = (df['CONDUCTA'].astype('string').str.strip().str.upper()
    .str.normalize('NFKD').str.encode('ascii',errors='ignore').str.decode('utf-8'))
cond_hosp = conducta.isin(['HOSPITALIZACION','HOSPITALIZACION PISO',
    'UNIDAD DE CUIDADOS INTENSIVOS','UCI'])

reglas = {
    'No hosp + fecha de hospitalizacion': pac_hosp.eq(2) & fecha_hos.notna(),
    'Hosp + sin fecha de hospitalizacion': pac_hosp.eq(1) & fecha_hos.isna(),
    'No hosp + conducta intrahospitalaria': pac_hosp.eq(2) & cond_hosp,
    'Inconsistencia estricta (3 condiciones)': pac_hosp.eq(2) & fecha_hos.notna() & cond_hosp,
}
tabla = pd.DataFrame([
    {'Regla':k,'N':int(v.sum()),'Pct':round(v.mean()*100,3)}
    for k,v in reglas.items()
])
print('AUDITORIA DE HOSPITALIZACION')
print('='*60)
display(tabla)

In [ ]:
# Correccion de inconsistencias
# Si un caso tiene fecha + conducta hospitalaria pero PAC_HOS_=2,
# se reclasifica como hospitalizado. La evidencia objetiva prevalece.
mascara = pac_hosp.eq(2) & fecha_hos.notna() & cond_hosp
n_corr = int(mascara.sum())
df.loc[mascara, 'PAC_HOS_'] = 1
print(f'Correccion: {n_corr} registros reclasificados como hospitalizados')

# Construccion del target binario (convencion ML: 1=positivo, 0=negativo)
df[TARGET_COL] = pd.to_numeric(df['PAC_HOS_'], errors='coerce').map({1:1,2:0}).astype('Int8')
dist = df[TARGET_COL].value_counts().sort_index()
for val, n in dist.items():
    label = 'Hospitalizado' if val==1 else 'No hospitalizado'
    print(f'  {val} ({label}): {n:,} ({n/len(df)*100:.1f}%)')

## 3. Correccion de CLASFINAL

Reglas de reclasificacion segun criterios INS:
- `COD_EVE=580` → MORTALIDAD POR DENGUE
- NaN + `COD_EVE=220` → DENGUE GRAVE
- NaN + `COD_EVE=210` + todos signos = 2 → DENGUE SIN SIGNOS

In [ ]:
# Correccion de CLASFINAL
df.loc[df['COD_EVE'].eq(580), 'CLASFINAL'] = 'MORTALIDAD POR DENGUE'
df.loc[df['CLASFINAL'].isna() & df['COD_EVE'].eq(220), 'CLASFINAL'] = 'DENGUE GRAVE'
df.loc[
    df['CLASFINAL'].isna() & df['COD_EVE'].eq(210)
    & df[SIGNOS_ALARMA].eq(2).all(axis=1),
    'CLASFINAL'
] = 'DENGUE SIN SIGNOS DE ALARMA'

print('Distribucion CLASFINAL (post-correccion):')
display(df['CLASFINAL'].value_counts(dropna=False)
    .rename_axis('Clasificacion').reset_index(name='N'))

## 4. Analisis temporal de hospitalizacion

> **Hallazgo critico:** Entre 2017-2020 existe una concentracion anormal de
> clasificaciones 'Con signos de alarma' (>95%) con tasa de hospitalizacion
> anomalamente baja (<45%). Desde 2021 supera el 83%.
> Esto sugiere ruptura estructural que debe controlarse en el modelado.

In [ ]:
# Hospitalizacion anual por clasificacion
hosp_anual = (
    df.groupby('ANO' if 'ANO' in df.columns else df.columns[df.columns.str.contains('O$|ANO')][0])
    [TARGET_COL].agg(Total='size', Hosp=lambda x: x.eq(1).sum())
    .reset_index()
)
# Ajustar nombre de columna de anio segun el CSV
col_anio = [c for c in df.columns if 'A' in c.upper() and ('O' in c or 'AO' in c)]
col_anio = col_anio[0] if col_anio else 'AÑO'

hosp_anual = (
    df.groupby(col_anio)[TARGET_COL]
    .agg(Total='size', Hosp=lambda x: x.eq(1).sum())
    .reset_index()
)
hosp_anual['Pct'] = (hosp_anual['Hosp'] / hosp_anual['Total'] * 100).round(2)

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(hosp_anual.iloc[:,0], hosp_anual['Pct'],
    marker='o', linewidth=2.5, markersize=8, color=COLORS['primary'])
for _, r in hosp_anual.iterrows():
    ax.text(r.iloc[0], r['Pct']+1.2, f"{r['Pct']:.1f}%",
        ha='center', fontsize=9, fontweight='bold')
pct_gral = df[TARGET_COL].eq(1).mean()*100
ax.axhline(pct_gral, linestyle='--', color=COLORS['neutral'],
    label=f'Promedio: {pct_gral:.1f}%')
ax.axvspan(2016.5, 2020.5, alpha=0.08, color='orange',
    label='Periodo con posible sesgo estructural')
ax.set_title('Evolucion anual de la hospitalizacion — Dengue Bucaramanga')
ax.set_xlabel('Ano')
ax.set_ylabel('Pacientes hospitalizados (%)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIGURES/'01_hospitalizacion_temporal.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 5. Calidad de variables clinicas

In [ ]:
# Completitud y prevalencia de variables clinicas
todas = SINTOMAS_GENERALES + SIGNOS_ALARMA
tabla_comp = pd.DataFrame(index=todas)
tabla_comp['Tipo'] = ['Sintoma general' if v in SINTOMAS_GENERALES
    else 'Signo de alarma' for v in todas]
tabla_comp['Pct_nulos']    = (df[todas].isna().mean()*100).round(2)
tabla_comp['Pct_presente'] = (df[todas].eq(1).mean()*100).round(2)
tabla_comp['Pct_ausente']  = (df[todas].eq(2).mean()*100).round(2)

print('Completitud variables clinicas:')
display(tabla_comp.sort_values('Pct_presente', ascending=False))
tabla_comp.to_csv(OUT_TABLES/'completitud_clinicas.csv')

# Grafico de prevalencia
df_p = tabla_comp.sort_values('Pct_presente')
colores = [COLORS['con_signos'] if t=='Signo de alarma'
    else COLORS['sin_signos'] for t in df_p['Tipo']]

fig, ax = plt.subplots(figsize=(11,6))
bars = ax.barh(df_p.index, df_p['Pct_presente'],
    color=colores, alpha=0.85, edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
leyenda = [mpatches.Patch(color=COLORS['con_signos'], label='Signo de alarma'),
    mpatches.Patch(color=COLORS['sin_signos'], label='Sintoma general')]
ax.legend(handles=leyenda, loc='lower right')
ax.set_xlabel('Prevalencia (%)')
ax.set_title('Prevalencia de manifestaciones clinicas del dengue — Bucaramanga')
ax.set_xlim(0,108)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIGURES/'02_prevalencia_sintomas.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 6. Guardado de checkpoints

Se guardan tres versiones para que los notebooks siguientes no repitan la limpieza.

In [ ]:
# Guardar checkpoints por estrato clinico
df_con = df[df['CLASFINAL'].eq('DENGUE CON SIGNOS DE ALARMA')].copy()
df_sin = df[df['CLASFINAL'].eq('DENGUE SIN SIGNOS DE ALARMA')].copy()

joblib.dump(df,     CHECKPOINTS/'01_df_completo.pkl')
joblib.dump(df_con, DATA_INTERIM/'datos_con_signos.pkl')
joblib.dump(df_sin, DATA_INTERIM/'datos_sin_signos.pkl')

print('Checkpoints guardados:')
print(f'  Completo : {df.shape[0]:,} registros')
print(f'  Con signos (DCSA): {df_con.shape[0]:,} | Hosp: {df_con[TARGET_COL].mean():.1%}')
print(f'  Sin signos (DSSA): {df_sin.shape[0]:,} | Hosp: {df_sin[TARGET_COL].mean():.1%}')

## Resumen EDA

| Hallazgo | Implicacion |
|----------|-------------|
| 2 inconsistencias corregidas en PAC_HOS_ | TARGET confiable |
| 16 casos mortalidad reclasificados | CLASFINAL consistente |
| Ruptura estructural 2017-2020 | Controlar anio en modelos |
| Fiebre prevalencia >95% | Alta colinealidad (VIF ~8) |
| DCSA: 65% hospitalizado | Desbalance moderado |
| DSSA: ~8% hospitalizado | Desbalance severo, usar AUC-PR |

**Siguiente:** `02_analisis_clinico/02a_analisis_univariado_bivariado.ipynb`